In [1]:
# 기본
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

# 데이터 전처리 알고리즘
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler

# 학습용과 검증용으로 나누는 함수
from sklearn.model_selection import train_test_split

# 교차 검증
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import cross_validate
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold

# 평가함수
# 분류용
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import roc_auc_score

# 회귀용
from sklearn.metrics import r2_score
from sklearn.metrics import mean_squared_error

# 모델의 최적의 하이퍼 파라미터를 찾기 위한 도구
from sklearn.model_selection import GridSearchCV

# 머신러닝 알고리즘 - 분류
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.ensemble import GradientBoostingClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier

# 머신러닝 알고리즘 - 회귀
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Ridge
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import GradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import VotingRegressor


# 학습 모델 저장을 위한 라이브러리
import pickle

### 프로젝트 셋팅

In [2]:
# 학습이 완료된 모델을 저장할 파일 이름
best_model_path = 'model/best_model_15_17_2.dat'
# 교차검증 횟수
cv_count = 10
# 교차 검증
kfold = KFold(n_splits=cv_count, shuffle=True, random_state=1)
# 평가 결과를 담을 리스트
# 필요하다면 다른 것도 만들어주세요
f1_score_list = []
# 학습 모델 이름
model_name_list = []

### 데이터 준비
- 데이터를 읽어오고 필요한 전처리까지 다 한다음 입력데이터는 train_X, 결과데이터는 train_y라는 변수에 담아서 준비해주세요

In [3]:
# 데이터를 읽어온다.
train_df = pd.read_parquet('Seleted(C_D)_delecteABE_all_train.parquet')
test_df = pd.read_parquet('Seleted(C_D)_delecteABE_all_test.parquet')

display(train_df)
display(test_df)

,기준년월,ID,Segment,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,Life_Stage,이용개월수_신용_R12M,이용개월수_신판_R12M,...,청구금액_R3M,청구금액_B0,청구서발송여부_B0,할인건수_R3M,할인건수_B0M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,46588,12226,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상,10회 이상
1,201807,TRAIN_000002,C,23988,24493,23988,1,자녀출산기,9,9,...,85931,21866,1,1회 이상,1회 이상,30회 이상,10회 이상,10회 이상,1회 이상,1회 이상
2,201807,TRAIN_000003,D,3904,5933,3904,1,자녀성장(2),12,12,...,61518,16356,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상,10회 이상
3,201807,TRAIN_000008,C,124967,68078,121279,5,자녀출산기,12,12,...,62715,20512,1,10회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
4,201807,TRAIN_000010,D,21001,18796,21001,1,자녀성장(1),12,12,...,30449,22512,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
476827,201812,TRAIN_399979,D,31187,27337,31187,2,자녀성장(2),12,12,...,41812,11817,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
476828,201812,TRAIN_399987,C,42492,35751,42492,1,자녀성장(2),12,12,...,68356,17859,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상,30회 이상
476829,201812,TRAIN_399993,C,72348,27792,72348,4,자녀성장(1),12,12,...,34890,10810,1,20회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상
476830,201812,TRAIN_399996,D,27636,26357,27636,1,자녀성장(2),12,12,...,37515,14402,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상


,기준년월,ID,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,Life_Stage,이용개월수_신용_R12M,이용개월수_신판_R12M,이용개월수_일시불_R12M,...,청구금액_R3M,청구금액_B0,청구서발송여부_B0,할인건수_R3M,할인건수_B0M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,201807,TEST_00000,21458,13852,21458,2,자녀성장(1),10,10,10,...,11441,4931,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
1,201807,TEST_00001,18681,11065,10759,2,자녀독립기,9,9,9,...,20522,10152,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
2,201807,TEST_00002,40758,27071,40758,2,자녀성장(1),12,12,12,...,50508,13223,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
3,201807,TEST_00003,5255,4827,5255,1,자녀성장(1),9,9,9,...,4604,2112,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
4,201807,TEST_00004,16148,8011,14290,3,자녀성장(1),12,12,12,...,6788,4406,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
599995,201812,TEST_99995,0,0,0,0,노년생활,0,0,0,...,0,0,0,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
599996,201812,TEST_99996,3110,1231,3110,1,자녀출산기,10,9,9,...,1256,359,1,1회 이상,1회 이상,10회 이상,1회 이상,1회 이상,1회 이상,1회 이상
599997,201812,TEST_99997,0,0,0,0,자녀성장(1),0,0,0,...,0,0,0,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
599998,201812,TEST_99998,173263,63592,113786,6,가족구축기,12,12,12,...,48141,21273,1,1회 이상,1회 이상,40회 이상,1회 이상,1회 이상,1회 이상,1회 이상


In [6]:
test_df.to_csv('C,D(17).csv', index=False, encoding='utf-8-sig')

In [7]:
# 데이터 프레임을 합친다.
all_df = pd.concat([train_df, test_df])
all_df.reset_index(inplace=True, drop=True)
all_df

,기준년월,ID,Segment,이용금액_R3M_신용체크,_1순위카드이용금액,이용금액_R3M_신용,이용카드수_신용체크,Life_Stage,이용개월수_신용_R12M,이용개월수_신판_R12M,...,청구금액_R3M,청구금액_B0,청구서발송여부_B0,할인건수_R3M,할인건수_B0M,방문횟수_앱_R6M,방문횟수_PC_R6M,방문일수_PC_R6M,인입횟수_ARS_R6M,이용메뉴건수_ARS_R6M
0,201807,TRAIN_000000,D,196,3681,196,1,자녀성장(2),12,12,...,46588,12226,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상,10회 이상
1,201807,TRAIN_000002,C,23988,24493,23988,1,자녀출산기,9,9,...,85931,21866,1,1회 이상,1회 이상,30회 이상,10회 이상,10회 이상,1회 이상,1회 이상
2,201807,TRAIN_000003,D,3904,5933,3904,1,자녀성장(2),12,12,...,61518,16356,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,10회 이상,10회 이상
3,201807,TRAIN_000008,C,124967,68078,121279,5,자녀출산기,12,12,...,62715,20512,1,10회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
4,201807,TRAIN_000010,D,21001,18796,21001,1,자녀성장(1),12,12,...,30449,22512,1,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1076827,201812,TEST_99995,NaN,0,0,0,0,노년생활,0,0,...,0,0,0,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
1076828,201812,TEST_99996,NaN,3110,1231,3110,1,자녀출산기,10,9,...,1256,359,1,1회 이상,1회 이상,10회 이상,1회 이상,1회 이상,1회 이상,1회 이상
1076829,201812,TEST_99997,NaN,0,0,0,0,자녀성장(1),0,0,...,0,0,0,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상,1회 이상
1076830,201812,TEST_99998,NaN,173263,63592,113786,6,가족구축기,12,12,...,48141,21273,1,1회 이상,1회 이상,40회 이상,1회 이상,1회 이상,1회 이상,1회 이상


In [8]:
all_df.drop(columns=['Segment','ID','기준년월'], inplace=True)

In [9]:
all_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1076832 entries, 0 to 1076831
Data columns (total 59 columns):
 #   Column              Non-Null Count    Dtype 
---  ------              --------------    ----- 
 0   이용금액_R3M_신용체크       1076832 non-null  int64 
 1   _1순위카드이용금액          1076832 non-null  int64 
 2   이용금액_R3M_신용         1076832 non-null  int64 
 3   이용카드수_신용체크          1076832 non-null  int64 
 4   Life_Stage          1076832 non-null  object
 5   이용개월수_신용_R12M       1076832 non-null  int64 
 6   이용개월수_신판_R12M       1076832 non-null  int64 
 7   이용개월수_일시불_R12M      1076832 non-null  int64 
 8   이용금액_일시불_R6M        1076832 non-null  int64 
 9   이용금액_일시불_B0M        1076832 non-null  int64 
 10  이용금액_일시불_R3M        1076832 non-null  int64 
 11  이용금액_일시불_R12M       1076832 non-null  int64 
 12  이용개월수_신용_R6M        1076832 non-null  int64 
 13  이용건수_신용_R6M         1076832 non-null  int64 
 14  이용건수_신용_B0M         1076832 non-null  int64 
 15  이용건수_신판_R6M         1076832 non-

In [10]:
# LabelEncoder 학습
Encoder1 = LabelEncoder()
Encoder2 = LabelEncoder()
Encoder3 = LabelEncoder()
Encoder4 = LabelEncoder()
Encoder5 = LabelEncoder()
Encoder6 = LabelEncoder()
Encoder7 = LabelEncoder()
Encoder8 = LabelEncoder()

Encoder1.fit(all_df['할인건수_R3M'])
Encoder2.fit(all_df['할인건수_B0M'])
Encoder3.fit(all_df['방문횟수_앱_R6M'])
Encoder4.fit(all_df['방문횟수_PC_R6M'])
Encoder5.fit(all_df['인입횟수_ARS_R6M'])
Encoder6.fit(all_df['Life_Stage'])
Encoder7.fit(all_df['이용메뉴건수_ARS_R6M'])
Encoder8.fit(all_df['방문일수_PC_R6M'])

LabelEncoder()

In [11]:
all_df['할인건수_R3M'] = Encoder1.transform(all_df['할인건수_R3M'])
all_df['할인건수_B0M'] = Encoder2.transform(all_df['할인건수_B0M'])
all_df['방문횟수_앱_R6M'] = Encoder3.transform(all_df['방문횟수_앱_R6M'])
all_df['방문횟수_PC_R6M'] = Encoder4.transform(all_df['방문횟수_PC_R6M'])
all_df['인입횟수_ARS_R6M'] = Encoder5.transform(all_df['인입횟수_ARS_R6M'])
all_df['Life_Stage'] = Encoder6.transform(all_df['Life_Stage'])
all_df['이용메뉴건수_ARS_R6M'] = Encoder7.transform(all_df['이용메뉴건수_ARS_R6M'])
all_df['방문일수_PC_R6M'] = Encoder8.transform(all_df['방문일수_PC_R6M'])

In [12]:
# Scaler 학습
scalerX = StandardScaler()
scalerX.fit(all_df)

,copy,True
,with_mean,True
,with_std,True


In [13]:
train_df['할인건수_R3M'] = Encoder1.transform(train_df['할인건수_R3M'])
train_df['할인건수_B0M'] = Encoder2.transform(train_df['할인건수_B0M'])
train_df['방문횟수_앱_R6M'] = Encoder3.transform(train_df['방문횟수_앱_R6M'])
train_df['방문횟수_PC_R6M'] = Encoder4.transform(train_df['방문횟수_PC_R6M'])
train_df['인입횟수_ARS_R6M'] = Encoder5.transform(train_df['인입횟수_ARS_R6M'])
train_df['Life_Stage'] = Encoder6.transform(train_df['Life_Stage'])
train_df['이용메뉴건수_ARS_R6M'] = Encoder7.transform(train_df['이용메뉴건수_ARS_R6M'])
train_df['방문일수_PC_R6M'] = Encoder8.transform(train_df['방문일수_PC_R6M'])

In [14]:
target1=pd.read_parquet(r'data/train/1.회원정보/201807_train_.parquet')
target2=pd.read_parquet(r'data/train/1.회원정보/201808_train_.parquet')
target3=pd.read_parquet(r'data/train/1.회원정보/201809_train_.parquet')
target4=pd.read_parquet(r'data/train/1.회원정보/201810_train_.parquet')
target5=pd.read_parquet(r'data/train/1.회원정보/201811_train_.parquet')
target6=pd.read_parquet(r'data/train/1.회원정보/201812_train_.parquet')

In [15]:
tg_df = pd.concat([
    target1['Segment'],
    target2['Segment'],
    target3['Segment'],
    target4['Segment'],
    target5['Segment'],
    target6['Segment']
])

tg_df = tg_df.reset_index(drop=True).to_frame(name='Segment')
tg_df = tg_df[tg_df['Segment'].isin(['C', 'D'])].reset_index(drop=True)
tg_df

,Segment
0,D
1,C
2,D
3,C
4,D
...,...
476827,D
476828,C
476829,C
476830,D


In [16]:
# 라벨 인코더 생성
le = LabelEncoder()

# 문자열 y를 숫자로 변환
tg_df['Segment'] = le.fit_transform(tg_df['Segment'])

In [17]:
# 입력과 결과로 나눈다.
X = train_df.drop(columns=['Segment','ID','기준년월'])
y = tg_df

In [18]:
# 표준화
X2 = scalerX.transform(X)
X2

array([[-0.94440155, -0.77288601, -0.88600734, ...,  0.07669481,
        -4.53088177, -3.58425652],
       [-0.167531  ,  0.37592125, -0.07071433, ..., -3.07285243,
         0.22070759,  0.09921615],
       [-0.82332572, -0.64857725, -0.75894335, ...,  0.07669481,
        -4.53088177, -3.58425652],
       ...,
       [ 1.41154849,  0.55802365,  1.58646329, ...,  0.07669481,
         0.22070759, -3.58425652],
       [-0.04841434,  0.47881269,  0.05429361, ...,  0.07669481,
         0.22070759,  0.09921615],
       [-0.19368573, -0.02824783, -0.09816262, ...,  0.07669481,
         0.22070759,  0.09921615]])

In [19]:
scaler_columns = X.columns.tolist()
scaler_columns

['이용금액_R3M_신용체크',
 '_1순위카드이용금액',
 '이용금액_R3M_신용',
 '이용카드수_신용체크',
 'Life_Stage',
 '이용개월수_신용_R12M',
 '이용개월수_신판_R12M',
 '이용개월수_일시불_R12M',
 '이용금액_일시불_R6M',
 '이용금액_일시불_B0M',
 '이용금액_일시불_R3M',
 '이용금액_일시불_R12M',
 '이용개월수_신용_R6M',
 '이용건수_신용_R6M',
 '이용건수_신용_B0M',
 '이용건수_신판_R6M',
 '이용건수_신용_R3M',
 '이용건수_신판_B0M',
 '이용건수_일시불_R6M',
 '이용건수_신판_R3M',
 '이용건수_일시불_B0M',
 '이용개월수_신판_R6M',
 '이용건수_일시불_R3M',
 '이용개월수_일시불_R6M',
 '이용후경과월_신판',
 '이용건수_신용_R12M',
 '이용건수_신판_R12M',
 '이용건수_일시불_R12M',
 '이용가맹점수',
 '이용후경과월_신용',
 '이용금액_오프라인_B0M',
 '이용금액_오프라인_R3M',
 '이용건수_오프라인_B0M',
 '_3순위업종_이용금액',
 '_3순위쇼핑업종_이용금액',
 '_2순위업종_이용금액',
 '이용개월수_오프라인_R6M',
 '이용금액_오프라인_R6M',
 '_2순위쇼핑업종_이용금액',
 '정상청구원금_B5M',
 '정상청구원금_B2M',
 '연속유실적개월수_기본_24M_카드',
 '정상청구원금_B0M',
 '정상입금원금_B0M',
 '정상입금원금_B5M',
 '정상입금원금_B2M',
 '이용개월수_전체_R6M',
 '이용개월수_전체_R3M',
 '청구금액_R6M',
 '청구금액_R3M',
 '청구금액_B0',
 '청구서발송여부_B0',
 '할인건수_R3M',
 '할인건수_B0M',
 '방문횟수_앱_R6M',
 '방문횟수_PC_R6M',
 '방문일수_PC_R6M',
 '인입횟수_ARS_R6M',
 '이용메뉴건수_ARS_R6M']

In [20]:
train_X = X2
train_y = y

### 기본 모델 사용하기
- 기본 모델 중에 만족하는 것을 찾았다면 하이퍼 파라미터 튜닝 과정은 생략하세요

In [21]:
# LGBM
lgbm_basic_model = LGBMClassifier(verbose=-1)
# 교차 검증을 수행한다
r1 = cross_val_score(lgbm_basic_model, train_X, train_y, scoring='f1', cv=kfold)
# 평가 결과를 담아준다.
f1_score_list.append(r1.mean())
# 학습 모델 이름을 담아준다.
model_name_list.append("LGBM Basic")

print(f'평균 f1 Score : {r1.mean()}')

평균 f1 Score : 0.8922105558173385


In [22]:
# XGBoost
xgboost_basic_model = XGBClassifier(verbose=-1, silent=True)
# 교차 검증을 수행한다
r1 = cross_val_score(xgboost_basic_model, train_X, train_y, scoring='f1', cv=kfold)
# 평가 결과를 담아준다.
f1_score_list.append(r1.mean())
# 학습 모델 이름을 담아준다.
model_name_list.append("XGBoost Basic")

print(f'평균 f1 Score : {r1.mean()}')

평균 f1 Score : 0.8975080409455203


In [23]:
d1 = {
    'f1 score' : f1_score_list
}
result_df = pd.DataFrame(d1, index=model_name_list)
result_df.sort_values(by='f1 score', ascending=False, inplace=True)
result_df

,f1 score
XGBoost Basic,0.897508
LGBM Basic,0.892211


---

In [24]:
best_model=xgboost_basic_model.fit(train_X, train_y)

In [25]:
with open(best_model_path, 'wb') as fp:
    pickle.dump(best_model, fp)
    pickle.dump(scalerX, fp)
    pickle.dump(scaler_columns, fp)
    pickle.dump(Encoder1, fp)
    pickle.dump(Encoder2, fp)
    pickle.dump(Encoder3, fp)
    pickle.dump(Encoder4, fp)
    pickle.dump(Encoder5, fp)
    pickle.dump(Encoder6, fp)
    pickle.dump(Encoder7, fp)
    pickle.dump(Encoder8, fp)
    pickle.dump(le, fp)

print('저장완료')

저장완료


---
### 혼동행렬